In [ ]:
import os
import cv2
import dlib
import imutils
import numpy as np
from imutils import face_utils
from collections import OrderedDict

In [ ]:
FACIAL_LANDMARKS_IDXS = OrderedDict([
    ("mouth", (48, 68)),
    ("inner_mouth", (60, 68)),
    ("right_eyebrow", (17, 22)),
    ("left_eyebrow", (22, 27)),
    ("right_eye", (36, 42)),
    ("left_eye", (42, 48)),
    ("nose", (27, 36)),
    ("jaw", (0, 17))
])

In [7]:
BASE_PATH = os.path.expanduser("~/100xDevs/personality/Personality-Classifier")

predictor_path = os.path.join(
    BASE_PATH,
    "models/landmark_model/shape_predictor_68_face_landmarks.dat"
)

detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(predictor_path)

print("✅ Dlib models loaded successfully.")

✅ Dlib models loaded successfully.


In [8]:
def get_landmarks(image):
    """
    Detects face and returns (shape, resized_image).
    Returns None if face not detected or multiple faces detected.
    """

    image = imutils.resize(image, width=600)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    rects = detector(gray, 1)

    if len(rects) != 1:
        return None, None

    shape = predictor(gray, rects[0])
    shape = face_utils.shape_to_np(shape)

    return shape, image

In [9]:
def extract_region(image, region_name, resize_width=250, padding=40):
    """
    Extract specific facial region (jaw, mouth, nose, etc.)
    """

    if region_name not in FACIAL_LANDMARKS_IDXS:
        raise ValueError("❌ Invalid region name")

    shape, image = get_landmarks(image)

    if shape is None:
        print("❌ No single face detected.")
        return None

    (start, end) = FACIAL_LANDMARKS_IDXS[region_name]

    (x, y, w, h) = cv2.boundingRect(np.array([shape[start:end]]))

    y_start = max(0, y)
    y_end = min(image.shape[0], y + h + padding)
    x_start = max(0, x)
    x_end = min(image.shape[1], x + w)

    roi = image[y_start:y_end, x_start:x_end]

    try:
        roi = imutils.resize(roi, width=resize_width)
        return roi
    except:
        return None

In [16]:
def process_folder(input_folder, output_folder, region_name):

    os.makedirs(output_folder, exist_ok=True)

    processed = 0
    skipped = 0

    for file in os.listdir(input_folder):

        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        input_path = os.path.join(input_folder, file)
        output_path = os.path.join(output_folder, file)

        image = cv2.imread(input_path)

        if image is None:
            skipped += 1
            continue

        roi = extract_region(image, region_name)

        if roi is not None:
            cv2.imwrite(output_path, roi)
            processed += 1
        else:
            skipped += 1

    print(f"✅ {region_name} extraction completed.")
    print(f"Processed: {processed}")
    print(f"Skipped: {skipped}")

In [19]:
region_name = "mouth"   # change to "jaw", "nose", "left_eye", etc.

input_folder = os.path.join(BASE_PATH, "sample_images")

output_folder = os.path.join(
    BASE_PATH,
    f"data/processed/sample/{region_name}"
)

process_folder(input_folder, output_folder, region_name)

❌ No single face detected.
✅ mouth extraction completed.
Processed: 7
Skipped: 1
